In [1]:
# ============================================================================
# IMPORT CENTRALIZED MODULES (Phase 1: Quick Wins)
# ============================================================================
import sys
sys.path.insert(0, '/Users/chloe/LLM-Based-Personal-Profile-Extraction')

from modules import (
    # Configuration
    PATHS,
    TASK1_DIRECT,
    TASK1_PSEUDOCODE,
    TASK1_ICL,
    PROMPT_STYLE_MAP,
    ABLATION_STYLES,
    EDUCATION_PROMPT,
    PipelineConfig,
    
    # Field definitions
    T1_FIELDS,
    T1_FIELDS_CMP,
    GT_FIELDS,
    
    # Patterns & hierarchies
    REGEX_PATTERNS,
    RELIGION_HIERARCHY,
    
    # Utilities
    NameNormalizer,
    HTMLProcessor,
    extract_readable_text,
    extract_infobox,
)

print("✓ All modules imported successfully")
print(f"  ├─ Paths: {len(PATHS)} configured")
print(f"  ├─ Prompts: {len(PROMPT_STYLE_MAP)} styles")
print(f"  ├─ Fields: {len(T1_FIELDS)} extraction fields")
print(f"  └─ Utilities: NameNormalizer, HTMLProcessor loaded")

✓ All modules imported successfully
  ├─ Paths: 12 configured
  ├─ Prompts: 3 styles
  ├─ Fields: 9 extraction fields
  └─ Utilities: NameNormalizer, HTMLProcessor loaded


# Senate Profile LLM Extraction Pipeline — v5
**DSBA 6010 — Chloe Partridge**

Aligned with Liu et al. (USENIX Security 2025) *Evaluating LLM-based Personal Information Extraction and Countermeasures*.

Key features:
- **Multi-provider support** — Groq (8B, 70B) and Gemini (configure in Section 2)
- **Prompt-style ablation** — direct, pseudocode, ICL (Section 4.2 / Table 13)
- **Religion signal annotation** — LLM-based pre-classification of bio text as `explicit` / `not_explicit` (Section 8b)
- **Traditional baselines** — regex + spaCy NER (Tables 4–5)
- **Evaluation metrics** — Accuracy, Rouge-1, BERT score with stratified religion analysis (Section 6.1.4)
- **Model comparison** — 8B vs 70B vs Gemini (Table 3 / Section 6.2)

**Quick Start:** Set your provider and model in Section 2, then run cells top to bottom.

## 1. Dependencies

In [2]:
# Install required packages for all supported providers
# !pip install beautifulsoup4 google-generativeai openai groq pandas tqdm rouge-score bert-score spacy
# !python -m spacy download en_core_web_sm

In [3]:
import json, time, re, random
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from groq import Groq
from rouge_score import rouge_scorer
import bert_score
import spacy

# Load spaCy model for baseline NER
nlp = spacy.load('en_core_web_sm')

/Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Configuration

Loads from `model_config_extraction.json`.

> **Before running:** if you have a `results_raw.json` from a previous run, rename it:
> ```bash
> mv senate_results/results_raw.json senate_results/results_raw_v1_backup.json
> ```


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# Section 2: Configuration (Updated to use centralized paths)
# ════════════════════════════════════════════════════════════════════════════

# Use centralized path configuration
HTML_DIR   = PATHS['html_data']
OUTPUT_DIR = PATHS['output_root']
config_path = PATHS['groq_config']

import os

if not config_path.exists():
    raise ValueError("groq_config_extraction.json not found at " + str(config_path))

with open(config_path) as f:
    config = json.load(f)

# Extract API key (from env var or config file)
api_key = os.getenv("GROQ_API_KEY") or config["api_key_info"]["api_keys"][0]

# Extract model and provider settings
provider = config["model_info"]["provider"]
model = config["model_info"]["name"]
temp = config["params"]["temperature"]
max_tok = config["params"]["max_output_tokens"]

# Initialize Groq client
from groq import Groq
api_client = Groq(api_key=api_key)

print(f"✓ Groq API initialized from config")
print(f"Model:       {model}")
print(f"Provider:    {provider}")
print(f"Temperature: {temp}  |  Max tokens: {max_tok}")

html_files = sorted(HTML_DIR.glob("*.html"))
print(f"HTML files:  {len(html_files)}")

✓ Groq API initialized from config
Model:       llama-3.1-8b-instant
Provider:    groq
Temperature: 0.1  |  Max tokens: 1500
HTML files:  100


## 2b. Session Metadata & Prompt Selection

Two modes for running the pipeline:

**Option 1: Single Prompt Style (default)**
- Set `RUN_ALL_PROMPT_STYLES = False`
- Choose a single style: `direct`, `pseudocode`, or `icl`
- Faster — one API call per senator
- Results in single-style column values

**Option 2: All Prompt Styles (new)**
- Set `RUN_ALL_PROMPT_STYLES = True`
- Automatically runs all three styles on every senator
- Slower: ~3 API calls per senator (with rate limiting)
- Results CSV will have 3× rows (one per prompt/senator combination)
- **Best for:** Prompt-style comparison on full dataset

**Liu et al. Context:**
- *Direct* vs *Pseudocode* (Table 13): pseudocode better for structured fields
- *Direct/Pseudocode* vs *ICL* (Figure 2): ICL best for occupation-type fields

All results will be recorded in `results_raw.json` and `task1_pii.csv` with `prompt_style` column for reproducibility.

In [5]:
import datetime

# ════════════════════════════════════════════════════════════════════════════
# Section 2b: Session Metadata & Prompt Selection (Using PipelineConfig)
# ════════════════════════════════════════════════════════════════════════════

# Initialize pipeline configuration
pipeline_config = PipelineConfig()

# Override any settings here if needed:
# pipeline_config.run_all_prompt_styles = False
# pipeline_config.active_prompt_style = "pseudocode"
# pipeline_config.inter_senator_delay = 10

RUN_ALL_PROMPT_STYLES = pipeline_config.run_all_prompt_styles
ACTIVE_PROMPT_STYLE = pipeline_config.active_prompt_style
STYLES_TO_RUN = pipeline_config.styles_to_run

# Log session metadata
session_metadata = pipeline_config.session_metadata

print("=" * 70)
print("📋 SESSION METADATA")
print("=" * 70)
for key, value in session_metadata.items():
    print(f"  {key:.<20s} {value}")
print("=" * 70)

session_info = f"Running {'all 3 prompt styles' if RUN_ALL_PROMPT_STYLES else f'single style: {ACTIVE_PROMPT_STYLE}'}"
print(f"\n✓ {session_info}\n")

📋 SESSION METADATA
  timestamp........... 2026-04-07T19:33:52.282072
  prompt_style........ all_styles
  run_all_styles...... True
  inter_senator_delay. 6
  between_styles_delay 3

✓ Running all 3 prompt styles



## 3. HTML Preprocessing

Strips `<script>`, `<style>`, `<nav>`, `<footer>` — noise with no informational value.  
Content preserved exactly as written.


In [6]:
def extract_readable_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s{2,}", " ", text).strip()

sample = html_files[0]
text   = extract_readable_text(sample.read_text(encoding="utf-8", errors="ignore"))
print(f"File:         {sample.name}")
print(f"Raw size:     {sample.stat().st_size:,} chars")
print(f"Cleaned size: {len(text):,} chars")
print(f"\n--- First 500 chars ---\n{text[:500]}")


File:         Adam_Schiff_CA.html
Raw size:     222,779 chars
Cleaned size: 7,918 chars

--- First 500 chars ---
About - Senator Schiff Skip to content Home Search Mobile Nav Open Open Search Close Search Contact Adam Close Mobile Nav Search About Senator Adam Schiff About Senator Adam Schiff After Adam graduated from law school, he moved to Los Angeles to serve as a law clerk for Judge William Matthew Byrne, Jr. Adam later joined the U.S. Attorney’s Office in Los Angeles as a federal prosecutor, where he served for almost six years, most notably prosecuting, Richard Miller, the first FBI agent ever to be 


## 4. Prompt Design

### Prompt style ablation (Liu et al. Section 4.2, Table 13)
Liu et al. test four styles: direct, persona, contextual, and pseudocode.
Pseudocode outperforms direct for structured fields (email, phone, name).
We implement both **direct** and **pseudocode** for Task 1 to replicate their finding.

### In-context learning (Liu et al. Section 6.2, Figure 2)
The paper shows ICL has the largest impact on occupation-type fields.
We add **one demonstration example** to Task 1 to test this on `committee_roles`, `education`, and `religious_affiliation`.

### Task 1 — Structured PII Extraction
Input: cleaned text.  
Extracts discrete fields: name, birthdate, gender, race, education, committee roles, and religious affiliation.


In [7]:
# ── Task 1 — DIRECT style (Liu et al. "direct" prompt, Table 13) ──────────
TASK1_DIRECT = """You are a precise data extraction specialist.
Extract the following fields from the senator profile text.
Return ONLY valid JSON. No preamble, no markdown fences.

{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "gender": string or null,
  "race_ethnicity": string or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "committee_roles": [string],
  "religious_affiliation": string or null,
  "religious_affiliation_inferred": boolean
}

Rules:
- full_name: Person's complete name. Extract directly from text.
- birthdate: Extract full date if available in YYYY-MM-DD format or MM/DD/YYYY format. Otherwise null.
- birth_year_inferred: Only if birthdate cannot be extracted but age/birth year is mentioned. Otherwise null.
- gender: Extract if explicitly stated OR inferable from pronouns (he/him → "male"). Otherwise null.
- race_ethnicity: Only if explicitly stated. Otherwise null. Do NOT infer.
- education: Array of objects with degree, institution, year. Include all entries found.
- committee_roles: Array of professional roles/committee memberships. Include all found.
- religious_affiliation: Use if mentioned explicitly OR inferred from organizational memberships, values, cultural references.
- religious_affiliation_inferred: Set to true if inferred based on signals; false if explicitly stated.
- Never guess; return null for missing fields
"""

# ── Task 1 — PSEUDOCODE style (Liu et al. Section 4.2, best for structured fields) ──
# The paper finds pseudocode achieves slightly better results for name, email,
# mailing address, and phone (Table 13). We apply it here for the analogous
# structured fields in the senator schema.
# 
# NOTE: Completely plain language version with zero code-like syntax to prevent
# the model from generating code instead of JSON. No function names or arrows.
TASK1_PSEUDOCODE = """You are a data extraction assistant. Follow these step-by-step instructions to extract information from the senator profile text below. Return ONLY valid JSON. No preamble, no markdown fences, no code.

Step 1 - Extract full_name: Find the senator's complete name directly from the text.
Step 2 - Extract birthdate: If a full date is present, format as YYYY-MM-DD. Otherwise null.
Step 3 - Extract birth_year_inferred: Only if birthdate is null AND a birth year or age is mentioned. Otherwise null.
Step 4 - Extract gender: Use explicit statement OR pronouns (he/him = "male", she/her = "female", they/them = "non-binary"). Otherwise null.
Step 5 - Extract race_ethnicity: Only if explicitly stated in the text. Do NOT infer. Otherwise null.
Step 6 - Extract education: Find all degree/institution/year entries. Return as array of objects.
Step 7 - Extract committee_roles: Find all committee memberships and professional roles. Return as array of strings.
Step 8 - Extract religious_affiliation: Use if explicitly stated OR inferable from organizational memberships, values language, or cultural references. Otherwise null.
Step 9 - Set religious_affiliation_inferred: true if inferred from signals, false if explicitly stated.

Return this exact JSON structure:
{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "gender": string or null,
  "race_ethnicity": string or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "committee_roles": [string],
  "religious_affiliation": string or null,
  "religious_affiliation_inferred": boolean
}

Never guess. Return null for missing fields.
"""

# ── Task 1 — DIRECT + IN-CONTEXT LEARNING (Liu et al. Section 6.2, Figure 2) ──
# The paper shows ICL has the largest impact on occupation-type fields.
# One demonstration example is provided for committee_roles and religious_affiliation.
TASK1_ICL = """You are a precise data extraction specialist.
Extract the following fields from the senator profile text.
Return ONLY valid JSON. No preamble, no markdown fences.

EXAMPLE INPUT:
Senator Jane Smith is from Ohio. She serves on the Senate Finance Committee and
the Senate Judiciary Committee. She earned a J.D. from Harvard Law School in 1995
and a B.A. from Ohio State University in 1992. She is known for her work on
interfaith initiatives and has spoken at numerous Christian conferences.

EXAMPLE OUTPUT:
{"full_name": "Jane Smith", "birthdate": null, "birth_year_inferred": null,
 "gender": "female", "race_ethnicity": null,
 "education": [{"degree": "J.D.", "institution": "Harvard Law School", "year": 1995},
               {"degree": "B.A.", "institution": "Ohio State University", "year": 1992}],
 "committee_roles": ["Senate Finance Committee", "Senate Judiciary Committee"],
 "religious_affiliation": "Christian", "religious_affiliation_inferred": true}

NOW EXTRACT:
{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "gender": string or null,
  "race_ethnicity": string or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "committee_roles": [string],
  "religious_affiliation": string or null,
  "religious_affiliation_inferred": boolean
}

Rules:
- full_name: Person's complete name. Extract directly from text.
- birthdate: Extract full date if available in YYYY-MM-DD format. Otherwise null.
- birth_year_inferred: Only if birthdate cannot be extracted but age/birth year is mentioned. Otherwise null.
- gender: Extract if explicitly stated OR inferable from pronouns (he/him → "male", she/her → "female", they/them → "non-binary"). Otherwise null.
- race_ethnicity: Only if explicitly stated. Otherwise null. Do NOT infer.
- education: Array of objects with degree, institution, year. Include all entries found.
- committee_roles: Array of professional roles/committee memberships. Include all found.
- religious_affiliation: Use if mentioned explicitly OR inferred from organizational memberships, values, cultural references.
- religious_affiliation_inferred: Set to true if inferred based on signals; false if explicitly stated.
- Never guess; return null for missing fields
"""

In [8]:
# ── Initialize session metadata (now that prompts are defined) ────────────
# Map style string to prompt template
PROMPT_STYLE_MAP = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# Validate selections
for style in STYLES_TO_RUN:
    if style not in PROMPT_STYLE_MAP:
        raise ValueError(f"Invalid style: {style}. Must be one of: {list(PROMPT_STYLE_MAP.keys())}")

# For session metadata, record mode
if RUN_ALL_PROMPT_STYLES:
    ACTIVE_PROMPT_NAME = "all_styles"
    session_info = f"Running all 3 prompt styles (direct, pseudocode, icl)"
else:
    ACTIVE_PROMPT_NAME = ACTIVE_PROMPT_STYLE
    session_info = f"Running single style: {ACTIVE_PROMPT_STYLE}"

# ── LOG SESSION METADATA ──────────────────────────────────────────────────
session_timestamp = datetime.datetime.now().isoformat()
session_metadata = {
    "timestamp": session_timestamp,
    "prompt_style": ACTIVE_PROMPT_NAME,
    "run_all_styles": RUN_ALL_PROMPT_STYLES,
    "model": model,
    "temperature": temp,
    "max_tokens": max_tok
}

print("=" * 70)
print("📋 SESSION METADATA")
print("=" * 70)
for key, value in session_metadata.items():
    print(f"  {key:.<20s} {value}")
print("=" * 70)
print(f"\n✓ {session_info}\n")

📋 SESSION METADATA
  timestamp........... 2026-04-07T19:33:52.316000
  prompt_style........ all_styles
  run_all_styles...... True
  model............... llama-3.1-8b-instant
  temperature......... 0.1
  max_tokens.......... 1500

✓ Running all 3 prompt styles (direct, pseudocode, icl)



In [9]:
def run_pipeline(text: str, model_override: str = None) -> dict:
    """
    Pipeline: extract PII from profile text using selected prompt style(s).
    Includes automatic retry on quota errors.
    
    Args:
        text: Senator profile text
        model_override: Optional model name for comparison
        
    Returns:
        If RUN_ALL_PROMPT_STYLES = False:
            Dict with keys:
            - task1_pii: Extraction results (single style)
            - prompt_style: Style used
            
        If RUN_ALL_PROMPT_STYLES = True:
            Dict with keys:
            - task1_pii: Contains all three style results
              {"direct": {...}, "pseudocode": {...}, "icl": {...}}
            - prompt_style: "all_styles" marker
    """
    if RUN_ALL_PROMPT_STYLES:
        # Run all three styles
        results = {}
        for style_name in ["direct", "pseudocode", "icl"]:
            prompt = PROMPT_STYLE_MAP[style_name]
            results[style_name] = call_groq(prompt, text, model_override=model_override, max_retries=5)
            # Increased rate limit between styles (3s) to reduce quota exhaustion
            time.sleep(3)
        
        return {
            "task1_pii": results,
            "prompt_style": "all_styles"
        }
    else:
        # Run single selected style
        prompt = PROMPT_STYLE_MAP[ACTIVE_PROMPT_STYLE]
        task1 = call_groq(prompt, text, model_override=model_override, max_retries=5)
        
        return {
            "task1_pii": task1,
            "prompt_style": ACTIVE_PROMPT_NAME
        }

In [10]:
def call_groq(prompt_template: str, text: str, model_override: str = None, max_retries: int = 5) -> dict:
    """
    Call LLM API with given prompt and text, parse JSON response.
    Supports multiple providers: Gemini, OpenAI (GPT), Groq, Llama.
    Includes exponential backoff retry logic for quota/rate limit errors.
    
    Args:
        prompt_template: Prompt string (can include {text} placeholder or expect text appended)
        text: Input text to extract from
        model_override: Optional model name override
        max_retries: Maximum number of retry attempts (default 5)
        
    Returns:
        Dict with extracted data (parsed from JSON response)
    """
    full_prompt = prompt_template + "\n\n" + text if "{text}" not in prompt_template else prompt_template.format(text=text)
    
    for attempt in range(max_retries):
        try:
            model_name = model_override or model
            response_text = None
            
            # Provider-specific API calls
            if provider.lower() == "groq":
                response = api_client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": full_prompt}],
                    temperature=temp,
                    max_tokens=max_tok
                )
                response_text = response.choices[0].message.content.strip()
            else:
                raise ValueError(f"Unsupported provider: {provider}. Only 'groq' is currently configured.")
            
            # Try to extract JSON from response (may have markdown fences)
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0].strip()
            elif "```" in response_text:
                response_text = response_text.split("```")[1].split("```")[0].strip()
            
            return json.loads(response_text)
        
        except json.JSONDecodeError as e:
            return {"error": "Failed to parse JSON response", "raw": response_text[:200] if response_text else "No response"}
        
        except Exception as e:
            error_str = str(e)
            
            # Check if it's a quota/rate limit error (common across all providers)
            is_quota_error = any([
                "RESOURCE_EXHAUSTED" in error_str,
                "quota" in error_str.lower(),
                "rate_limit" in error_str.lower(),
                "rate limit" in error_str.lower(),
                "too many requests" in error_str.lower(),
                "please retry" in error_str.lower(),
                "429" in error_str,  # HTTP 429
                "503" in error_str,  # HTTP 503
            ])
            
            if is_quota_error and attempt < max_retries - 1:
                # Extract retry delay if available
                retry_match = re.search(r'retry in (\d+\.\d+)', error_str)
                if retry_match:
                    wait_time = float(retry_match.group(1)) + 2  # Add 2 second buffer
                else:
                    # Exponential backoff: 2^attempt seconds, max 120 seconds
                    wait_time = min(2 ** attempt, 120)
                
                print(f"⏳ Rate limit hit (attempt {attempt+1}/{max_retries}). Waiting {wait_time:.1f}s before retry...")
                time.sleep(wait_time)
                continue
            else:
                # Either not a quota error or we've exhausted retries
                return {"error": str(e)}

In [11]:
# Test on just one senator with full debugging
print(f"Testing {provider.upper()} API with retry logic...\n")
test_html = html_files[0]
test_text = extract_readable_text(test_html.read_text(encoding="utf-8", errors="ignore"))

print(f"Sample file: {test_html.name}")
print(f"Text length: {len(test_text)} chars\n")

try:
    # Call through the retry-enabled function
    result = call_groq(TASK1_DIRECT, test_text)
    
    if "error" in result:
        print(f"❌ Extraction error: {result['error']}")
        if "raw" in result:
            print(f"Response snippet: {result['raw']}")
    else:
        print("✓ Successfully extracted and parsed JSON")
        print(json.dumps(result, indent=2))
        
except Exception as e:
    print(f"❌ Exception occurred: {e}")
    import traceback
    traceback.print_exc()


Testing GROQ API with retry logic...

Sample file: Adam_Schiff_CA.html
Text length: 7918 chars

✓ Successfully extracted and parsed JSON
{
  "full_name": "Adam Schiff",
  "birthdate": null,
  "birth_year_inferred": null,
  "gender": "male",
  "race_ethnicity": null,
  "education": [
    {
      "degree": null,
      "institution": "Stanford University",
      "year": null
    },
    {
      "degree": null,
      "institution": "Harvard Law School",
      "year": null
    }
  ],
  "committee_roles": [
    "Chairman of the House Permanent Select Committee on Intelligence",
    "Chairman of the House Select Committee to Investigate the January 6th Attack on the U.S. Capitol",
    "Member of the House Select Committee on Benghazi",
    "Member of the House Committee on the Judiciary",
    "Member of the House Committee on Foreign Affairs",
    "Member of the House Appropriations Committee"
  ],
  "religious_affiliation": null,
  "religious_affiliation_inferred": false
}


## 5. Run Pipeline

Single API call per senator for Task 1 PII extraction.  
Results saved incrementally — safe to interrupt and resume.


In [12]:
raw_path = OUTPUT_DIR / "results_raw.json"

if raw_path.exists():
    with open(raw_path) as f:
        results = json.load(f)
    done_ids = {r["senator_id"] for r in results}
    print(f"Resuming — {len(done_ids)} senators already processed.")
else:
    results, done_ids = [], set()

remaining = [f for f in html_files if f.stem not in done_ids]
print(f"Senators remaining: {len(remaining)}")

# More aggressive rate limiting to avoid quota exhaustion
# When running all 3 prompt styles, each senator gets 3 API calls
# With 3-second delays between styles + main intercall delay = safer
INTER_SENATOR_DELAY = 6 if RUN_ALL_PROMPT_STYLES else 4

print(f"Rate limit: {INTER_SENATOR_DELAY}s between senators\n")

for html_file in tqdm(remaining, desc="Processing senators"):
    senator_id = html_file.stem
    html       = html_file.read_text(encoding="utf-8", errors="ignore")
    text       = extract_readable_text(html)
    result     = run_pipeline(text)

    results.append({"senator_id": senator_id, **result})
    with open(raw_path, "w") as f:
        json.dump(results, f, indent=2)

    time.sleep(INTER_SENATOR_DELAY)

print(f"\nDone. {len(results)} senators processed.")


Resuming — 100 senators already processed.
Senators remaining: 0
Rate limit: 6s between senators



Processing senators: 0it [00:00, ?it/s]


Done. 100 senators processed.


In [13]:
from bs4 import BeautifulSoup
from pathlib import Path

html = (HTML_DIR / "Amy_Klobuchar_MN.html").read_text(encoding="utf-8", errors="ignore")
soup = BeautifulSoup(html, "html.parser")
for tag in soup(["script", "style", "nav", "footer", "noscript"]):
    tag.decompose()
print(soup.get_text(separator=" ", strip=True)[:1000])

About Amy - U.S. Senator Amy Klobuchar Menu Search Search Amy Klobuchar U.S. Senator for Minnesota About Amy Share this on Facebook Share this on Twitter Home About Amy Print Senator Klobuchar, joined by her family, is sworn into her fourth term in the U.S. Senate by Vice President Kamala Harris. U.S. Senator Amy Klobuchar is the first woman elected to represent the State of Minnesota in the United States Senate. Throughout her public service, Senator Klobuchar has always embraced the values she learned growing up in Minnesota. Her grandfather worked 1500 feet underground in the iron ore mines of Northern Minnesota. Her father, Jim, was a newspaperman, and her mother, Rose, was an elementary school teacher who continued teaching until she was 70. Senator Klobuchar has built a reputation of putting partisanship aside to help strengthen the economy and support families, workers, and businesses. In the 117th Congress, she was number one in the Senate for introducing bipartisan bills and n

## 6. Flatten Results to CSV

- `task1_pii.csv` — structured PII fields  

Safe to run mid-pipeline to inspect partial results.



In [18]:
with open(OUTPUT_DIR / "results_raw.json") as f:
    results = json.load(f)

# Flatten Task 1 results to CSV
task1_rows = []
for r in results:
    t1_data = r.get("task1_pii", {})
    prompt_style = r.get("prompt_style", "unknown")
    
    # Handle both single-style and all-styles results
    if prompt_style == "all_styles":
        # All three styles were run — create separate rows for each style
        for style_name, style_result in t1_data.items():
            row = {
                "senator_id": r["senator_id"],
                "prompt_style": style_name,
                "extraction_error": style_result.get("error")
            }
            for field in T1_FIELDS:
                row[field] = style_result.get(field)
            task1_rows.append(row)
    else:
        # Single style was run
        row = {
            "senator_id": r["senator_id"],
            "prompt_style": prompt_style,
            "extraction_error": t1_data.get("error")
        }
        for field in T1_FIELDS:
            row[field] = t1_data.get(field)
        task1_rows.append(row)

df_t1 = pd.DataFrame(task1_rows)
df_t1.to_csv(OUTPUT_DIR / "task1_pii.csv", index=False)
print(f"Saved {len(df_t1)} rows to task1_pii.csv")
print(f"\nPrompt styles in results:")
print(df_t1["prompt_style"].value_counts().to_string())

Saved 300 rows to task1_pii.csv

Prompt styles in results:
prompt_style
direct        100
pseudocode    100
icl           100


## XX Traditional Baselines (Liu et al. Tables 4–5)

Liu et al. benchmark LLMs against regex, keyword search, and spaCy NER,
finding LLMs outperform traditional methods in almost all scenarios.
We implement the two most relevant baselines for our senator schema:
- **Regex** — structured fields (name, party via keyword, education year)
- **spaCy NER** — name (PERSON), affiliation/committee (ORG)

Running these on the same profiles lets us reproduce the LLM-vs-traditional comparison
in our own dataset as described in their Section 6.2.

In [ ]:
# ── Regex baseline (Liu et al. Section 6.1.3) ────────────────────────────
# Mirrors the paper's regular expression approach for fields with clear structure.
import re as _re

# Unpack regex patterns from centralized config
NAME_RE = REGEX_PATTERNS['NAME_RE']
PARTY_KEYWORD_RE = REGEX_PATTERNS['PARTY_KEYWORD_RE']
YEAR_RE = REGEX_PATTERNS['YEAR_RE']
EMAIL_RE = REGEX_PATTERNS['EMAIL_RE']
PHONE_RE = REGEX_PATTERNS['PHONE_RE']

def regex_extract(text: str) -> dict:
    """Regex baseline — structured fields only."""
    name_m  = NAME_RE.search(text)
    party_m = PARTY_KEYWORD_RE.search(text)
    years   = YEAR_RE.findall(text)
    return {
        "full_name":  name_m.group(1) if name_m else None,
        "party":      party_m.group(0).title() if party_m else None,
        "email":      EMAIL_RE.findall(text) or None,
        "phone":      PHONE_RE.findall(text) or None,
        "years_found": years[:5] if years else None,
    }

# ── spaCy NER baseline (Liu et al. Section 6.1.3) ────────────────────────
def spacy_extract(text: str) -> dict:
    """spaCy NER baseline — PERSON, ORG entities."""
    doc = nlp(text[:10000])  # spaCy token limit safety
    persons = list({ent.text for ent in doc.ents if ent.label_ == "PERSON"})
    orgs    = list({ent.text for ent in doc.ents if ent.label_ == "ORG"})
    return {
        "persons_detected": persons[:5],
        "orgs_detected":    orgs[:10],
    }

# ── Quick smoke test on first profile ────────────────────────────────────
sample_html = html_files[0].read_text(encoding="utf-8", errors="ignore")
sample_text = extract_readable_text(sample_html)
print(f"Baseline test on: {html_files[0].name}")
print("=== Regex baseline ===")
print(json.dumps(regex_extract(sample_text), indent=2))
print("\n=== spaCy NER baseline ===")
print(json.dumps(spacy_extract(sample_text), indent=2))

Baseline test on: Adam_Schiff_CA.html
=== Regex baseline ===


NameError: name 'NAME_RE' is not defined

## 7. Quick Descriptive Analysis

In [17]:
print("=== Task 1: Field Coverage ===")
for col in ["full_name","birthdate","birth_year_inferred","gender","race_ethnicity","education","committee_roles","religious_affiliation","religious_affiliation_inferred"]:
    filled = df_t1[col].notna().sum()
    print(f"  {col:40s}: {filled}/{len(df_t1)}")

=== Task 1: Field Coverage ===
  full_name                               : 300/300
  birthdate                               : 51/300
  birth_year_inferred                     : 0/300
  gender                                  : 267/300
  race_ethnicity                          : 17/300
  education                               : 300/300
  committee_roles                         : 299/300
  religious_affiliation                   : 97/300
  religious_affiliation_inferred          : 300/300


## 8. Ground Truth Builder — Multi-Source Pipeline

Builds **ground_truth.csv** by combining data from:
1. **Wikipedia** — birthdate, gender, race_ethnicity
2. **Ballotpedia** — committee_roles
3. **Pew Research** — religion (via fuzzy matching)

**Input:** senators_index.csv with full names and states  
**Intermediate:** senators_index.csv is enriched with Wikipedia and Ballotpedia URLs  
**Output:** senate_ground_truth.csv with columns: name, state, full_name, birthdate, gender, race_ethnicity, committee_roles, religion

**Features:**
- Resume-safe: incremental saving every 10 senators
- Comprehensive logging to scrape_errors.log
- Fuzzy matching (rapidfuzz) against Pew religion data
- Connection pooling via requests.Session() for efficient scraping


In [20]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
import os
import logging
from rapidfuzz import process, fuzz

INPUT_PATH = "../external_data/senate_html/senators_index.csv"
PEW_PATH = "../external_data/ground_truth/pew_religion.csv"
OUTPUT_PATH = "../external_data/ground_truth/senate_ground_truth.csv"
LOG_PATH = "../external_data/ground_truth/scrape_errors.log"

os.makedirs("../external_data/ground_truth", exist_ok=True)
logging.basicConfig(filename=LOG_PATH, level=logging.WARNING,
                    format="%(asctime)s %(message)s")

In [21]:
# ── Load senators_index.csv and construct URLs ──────────────────────────
senators = pd.read_csv(INPUT_PATH)

# Instead of: create_slug(name) for each row
# Use: NameNormalizer methods for better name normalization
senators["name_clean"] = senators["name"].str.replace(r'\s+[A-Z]\.\s*', ' ', regex=True).str.strip()
senators["wikipedia_url"] = senators["name"].apply(NameNormalizer.create_wikipedia_url)
senators["ballotpedia_url"] = senators["name"].apply(NameNormalizer.create_ballotpedia_url)

print(f"Loaded {len(senators)} senators from {INPUT_PATH}")
print(f"\nSample URLs:")
print(senators[["name", "wikipedia_url", "ballotpedia_url"]].head(3).to_string())

Loaded 99 senators from ../external_data/senate_html/senators_index.csv

Sample URLs:
               name                                    wikipedia_url                           ballotpedia_url
0       Katie Britt    https://en.wikipedia.org/wiki/katherine_britt   https://ballotpedia.org/katherine_britt
1  Tommy Tuberville  https://en.wikipedia.org/wiki/Thomas_Tuberville  https://ballotpedia.org/tommy_tuberville
2    Lisa Murkowski     https://en.wikipedia.org/wiki/lisa_murkowski    https://ballotpedia.org/lisa_murkowski


In [22]:
def scrape_wikipedia(url, name):
    """
    Scrape Wikipedia for a senator's profile.
    
    Args:
        url: Wikipedia URL for the senator
        name: Senator's name for logging
        
    Returns:
        dict with keys: full_name, birthdate, gender, race_ethnicity
        On failure, all fields are returned as NaN
    """
    result = {
        "full_name": None,
        "birthdate": None,
        "gender": None,
        "race_ethnicity": None
    }
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Extract full_name from page heading (h1)
        h1_tag = soup.find("h1")
        if h1_tag:
            page_title = h1_tag.get_text(strip=True)
            # Remove disambiguation suffix if present
            result["full_name"] = re.sub(r'\s*\(.*?\)\s*$', '', page_title).strip()
        
        # Find infobox (table with class containing "infobox")
        infobox = soup.find("table", {"class": lambda x: x and "infobox" in x})
        
        if infobox:
            rows = infobox.find_all("tr")
            
            for row in rows:
                cells = row.find_all(["th", "td"])
                if len(cells) >= 2:
                    label = cells[0].get_text(strip=True).lower()
                    value = cells[1].get_text(strip=True)
                    
                    # Extract birthdate from "Born" row
                    if label.startswith("born") or label.startswith("birth"):
                        # Try to extract date pattern like "Month DD, YYYY" or "YYYY-MM-DD"
                        date_match = re.search(
                            r'(\w+\s+\d{1,2},?\s+\d{4}|\d{4}-\d{2}-\d{2})',
                            value
                        )
                        if date_match:
                            result["birthdate"] = date_match.group(0)
                    
                    # Extract race_ethnicity (explicit mention only)
                    if label in ["ethnicity", "race", "nationality"]:
                        result["race_ethnicity"] = value
        
        # If gender not found in infobox, check first paragraph for pronouns
        if not result["gender"]:
            for p in soup.find_all("p"):
                text = p.get_text()
                if len(text) > 50 and any(
                    keyword in text.lower() for keyword in ["is a", "was a", "american", "senator"]
                ):
                    text_lower = text.lower()
                    if any(p in text_lower for p in ["she ", "her ", "hers"]):
                        result["gender"] = "Female"
                    elif any(p in text_lower for p in ["he ", "him ", "his "]):
                        result["gender"] = "Male"
                    break
        
    except Exception as e:
        logging.warning(f"Wikipedia scrape failed for {name}: {str(e)}")
    
    return result

In [23]:
def scrape_ballotpedia(url, name):
    """
    Scrape Ballotpedia for a senator's committee assignments.
    
    Args:
        url: Ballotpedia URL for the senator
        name: Senator's name for logging
        
    Returns:
        dict with key: committee_roles (pipe-delimited string)
        On failure, committee_roles is set to NaN
    """
    result = {
        "committee_roles": None
    }
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Find "Committee assignments" h2
        committee_heading = None
        for h in soup.find_all("h2"):
            if "committee assignments" in h.get_text().lower():
                committee_heading = h
                break

        if committee_heading:
            current = committee_heading.find_next("h4")
            while current:
                if "2025-2026" in current.get_text():
                    committee_roles = []
                    sib = current.find_next()
                    while sib and sib.name not in ["h4", "h3", "h2"]:
                        if sib.name == "li":
                            text = sib.get_text(strip=True)
                            if text:
                                committee_roles.append(text)
                        sib = sib.find_next()
                    if committee_roles:
                        result["committee_roles"] = "|".join(committee_roles)
                    break
                current = current.find_next("h4")
        
    except Exception as e:
        logging.warning(f"Ballotpedia scrape failed for {name}: {str(e)}")
    
    return result

In [31]:
def merge_pew(senators_df, pew_path):
    """
    Merge Pew religion data using fuzzy name matching.

    Args:
        senators_df: DataFrame with column 'name'
        pew_path: Path to Pew CSV with columns: name, state, religion

    Returns:
        Series of religion values aligned to senators_df index
        Returns all None if pew_path file doesn't exist
    """
    religion_series = pd.Series(index=senators_df.index, dtype=object)
    religion_series[:] = None

    # ── Senators with no Pew entry (appointed after Pew data was collected) ──
    # These will never match correctly — force null rather than accept bad match.
    NO_PEW_ENTRY = {
        "Alan Armstrong",   # appointed March 2026
        "Ashley Moody",     # appointed January 2025
        "Jon Husted",       # appointed January 2025
    }

    # ── Nickname → Pew formal name mapping ───────────────────────────────────
    # Pew uses formal/legal names; senate bios often use nicknames.
    NAME_ALIASES = {
        "Chuck Grassley":  "Charles Grassley",
        "Chuck Schumer":   "Charles Schumer",
        "Dick Durbin":     "Richard Durbin",
        "Ed Markey":       "Edward Markey",
        "Chris Murphy":    "Christopher Murphy",
        "Mike Crapo":      "Michael Crapo",
    }

    # Check if Pew CSV exists
    if not os.path.exists(pew_path):
        logging.warning(f"Pew religion CSV not found at {pew_path} — skipping merge")
        print(f"⚠️  Pew religion file not found: {pew_path}")
        print(f"   Religion column will be empty. To populate:")
        print(f"   1. Create {pew_path} with columns: name, state, religion")
        print(f"   2. Re-run this cell\n")
        return religion_series

    try:
        pew_df = pd.read_csv(pew_path)
        pew_names = pew_df["name"].tolist()
        print(pew_df.head(2))
        print(pew_names[:3])

        for idx, row in senators_df.iterrows():
            senator_name = row["name"]

            # Skip senators with no Pew entry
            if senator_name in NO_PEW_ENTRY:
                logging.info(f"No Pew entry for {senator_name} — skipping")
                continue

            # Apply alias if nickname is used
            lookup_name = NAME_ALIASES.get(senator_name, senator_name)

            # Fuzzy match against Pew names
            best_match, score, _ = process.extractOne(
                lookup_name, pew_names, scorer=fuzz.token_sort_ratio
            )

            if score >= 85:
                pew_match = pew_df[pew_df["name"] == best_match].iloc[0]
                religion_series.loc[idx] = pew_match["religion"]
            else:
                logging.warning(f"Pew match failed for {senator_name} (score={score})")

    except Exception as e:
        logging.warning(f"Error merging Pew data: {str(e)}")
        print(f"⚠️  Error reading Pew CSV: {str(e)}\n")

    return religion_series


In [32]:
# ── Main scrape loop: resume-safe with incremental saving ────────────────
# Load existing results if file exists
if os.path.exists(OUTPUT_PATH):
    df_existing = pd.read_csv(OUTPUT_PATH)
    completed = set(df_existing["name"].values)
    print(f"Found existing ground_truth.csv with {len(completed)} entries. Resuming...")
else:
    completed = set()
    print(f"Creating new ground_truth.csv")

results = []
session = requests.Session()

# Define CSV columns
columns = ["name", "state", "full_name", "birthdate", "gender", "race_ethnicity",
           "committee_roles", "religion"]

# Initialize CSV with header if new file
if not os.path.exists(OUTPUT_PATH):
    df_empty = pd.DataFrame(columns=columns)
    df_empty.to_csv(OUTPUT_PATH, index=False)

print(f"\n{'='*70}")
print("🔄 Starting multi-source ground truth scrape")
print(f"{'='*70}\n")

for idx, row in senators.iterrows():
    name = row["name"]
    state = row["state"]
    
    # Skip if already scraped
    if name in completed:
        continue
    
    # Scrape Wikipedia
    wiki_result = scrape_wikipedia(row["wikipedia_url"], name)
    time.sleep(1)
    
    # Scrape Ballotpedia
    ballotpedia_result = scrape_ballotpedia(row["ballotpedia_url"], name)
    time.sleep(1.5)
    
    # Combine results
    combined = {
        "name": name,
        "state": state,
        "full_name": wiki_result.get("full_name"),
        "birthdate": wiki_result.get("birthdate"),
        "gender": wiki_result.get("gender"),
        "race_ethnicity": wiki_result.get("race_ethnicity"),
        "committee_roles": ballotpedia_result.get("committee_roles"),
        "religion": None  # Will be filled in next step
    }
    
    results.append(combined)
    completed.add(name)
    
    # Save every 10 senators (incremental save — resume-safe)
    if len(results) >= 10:
        df_batch = pd.DataFrame(results)
        df_batch.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
        results = []
        
        # Print progress
        print(f"✓ Scraped 10/{len(senators)}: {name}")

# Write remaining results
if results:
    df_batch = pd.DataFrame(results)
    df_batch.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)

print(f"\n{'='*70}")
print(f"✓ Scraping complete: {len(completed)} senators")
print(f"{'='*70}\n")

Found existing ground_truth.csv with 99 entries. Resuming...

🔄 Starting multi-source ground truth scrape


✓ Scraping complete: 99 senators



In [33]:
# ── Merge Pew religion data and print coverage summary ───────────────────
# Load the saved ground truth CSV
df_gt = pd.read_csv(OUTPUT_PATH)

df_gt["senator_id"] = df_gt["name"].str.replace(" ", "_") + "_" + df_gt["state"]

# Merge Pew religion data (will return all None if file doesn't exist)
religion_series = merge_pew(df_gt, PEW_PATH)
df_gt["religion"] = religion_series.values

# Save updated ground truth with religion column
df_gt.to_csv(OUTPUT_PATH, index=False)

print("\n" + "="*70)
print("📊 GROUND TRUTH COVERAGE SUMMARY")
print("="*70)
print(f"Total senators scraped: {len(df_gt)}\n")

coverage_fields = ["full_name", "birthdate", "gender", "race_ethnicity",
                   "committee_roles", "religion"]
coverage_stats = []

for field in coverage_fields:
    # Count non-null, non-NaN values
    non_null = df_gt[field].notna().sum()
    pct = (non_null / len(df_gt)) * 100 if len(df_gt) > 0 else 0
    coverage_stats.append({
        "Field": field,
        "Non-Null": non_null,
        "Coverage": f"{pct:.1f}%"
    })

df_coverage = pd.DataFrame(coverage_stats)
print(df_coverage.to_string(index=False))
print("="*70)
print(f"\n✓ Final output saved to: {OUTPUT_PATH}")
print(f"✓ Errors logged to: {LOG_PATH}")
if os.path.exists(PEW_PATH):
    print(f"✓ Religion data merged from: {PEW_PATH}\n")
else:
    print(f"⚠️  Religion column empty (Pew CSV not found)\n")

             name state  religion
0  Lisa Murkowski    AK  Catholic
1    Dan Sullivan    AK  Catholic
['Lisa Murkowski', 'Dan Sullivan', 'Katie Britt']

📊 GROUND TRUTH COVERAGE SUMMARY
Total senators scraped: 99

          Field  Non-Null Coverage
      full_name        99   100.0%
      birthdate        96    97.0%
         gender        96    97.0%
 race_ethnicity         0     0.0%
committee_roles        79    79.8%
       religion        96    97.0%

✓ Final output saved to: ../external_data/ground_truth/senate_ground_truth.csv
✓ Errors logged to: ../external_data/ground_truth/scrape_errors.log
✓ Religion data merged from: ../external_data/ground_truth/pew_religion.csv



## 8b. Religion Signal Annotation

Classifies each senator's bio text as `explicit` (religion directly mentioned)
or `not_explicit` (absent or only inferable from indirect signals).

This is an **input characterisation step**, not ground truth annotation — it
describes what information was available to the LLM, not what the correct answer is.
It runs as a separate API call to avoid contaminating the main extraction task.

**Why this matters:** If the model achieves high accuracy on `religious_affiliation`,
we need to know how much of that comes from bios where religion was stated outright
versus bios requiring multi-hop inference. These are different claims about LLM capability.

**Output:** Adds `religion_signal` column to `senate_ground_truth.csv`.
Values: `explicit` | `not_explicit` | `error`

**Spot-check:** Review ~15–20 cases manually against the raw bio text before
drawing conclusions from stratified accuracy numbers.

In [34]:
# ── Load senators_index.csv and construct URLs ──────────────────────────
senators = pd.read_csv(INPUT_PATH)

senators["name_clean"] = senators["name"].str.replace(r'\s+[A-Z]\.\s*', ' ', regex=True).str.strip()
senators["wikipedia_url"] = senators["name"].apply(NameNormalizer.create_wikipedia_url)
senators["ballotpedia_url"] = senators["name"].apply(NameNormalizer.create_ballotpedia_url)

print(f"Loaded {len(senators)} senators from {INPUT_PATH}")
print(f"\nSample URLs:")
print(senators[["name", "wikipedia_url", "ballotpedia_url"]].head(3).to_string())

Loaded 99 senators from ../external_data/senate_html/senators_index.csv

Sample URLs:
               name                                    wikipedia_url                           ballotpedia_url
0       Katie Britt    https://en.wikipedia.org/wiki/katherine_britt   https://ballotpedia.org/katherine_britt
1  Tommy Tuberville  https://en.wikipedia.org/wiki/Thomas_Tuberville  https://ballotpedia.org/tommy_tuberville
2    Lisa Murkowski     https://en.wikipedia.org/wiki/lisa_murkowski    https://ballotpedia.org/lisa_murkowski


In [35]:
# ── Education ground truth extraction from Wikipedia ─────────────────────
# Uses Wikipedia page text + LLM extraction, saved for manual review.
# After running, manually verify education_text column before using in evaluation.

EDUCATION_PROMPT = """You are a precise data extraction specialist.
Extract the education history from the text below.
Return ONLY a pipe-delimited string of degree|institution|year entries.
Example: "B.A.|Stanford University|1982|J.D.|Harvard Law School|1986"
If no education found, return null.
Do not explain. Output only the string or null.
"""


df_gt = pd.read_csv(OUTPUT_PATH)

if "education_text" not in df_gt.columns:
    df_gt["education_text"] = None

needs_education = df_gt[df_gt["education_text"].isna()]
print(f"Senators needing education extraction: {len(needs_education)}")

for idx, row in tqdm(needs_education.iterrows(), total=len(needs_education),
                     desc="Extracting education"):
    senator_name = row["name"]
    wiki_url = create_wikipedia_url(senator_name)

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(wiki_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        wiki_text = soup.get_text(separator=" ", strip=True)[:5000]
    except Exception as e:
        logging.warning(f"Wikipedia fetch failed for {senator_name} ({wiki_url}): {e}")
        # Try fallback: direct space->underscore substitution (may work for some redirects)
        try:
            fallback_url = "https://en.wikipedia.org/wiki/" + senator_name.replace(" ", "_")
            response = requests.get(fallback_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")
            for tag in soup(["script", "style", "nav", "footer"]):
                tag.decompose()
            wiki_text = soup.get_text(separator=" ", strip=True)[:5000]
        except Exception as e2:
            logging.warning(f"Fallback Wikipedia fetch also failed for {senator_name}: {e2}")
            df_gt.at[idx, "education_text"] = "error"
            df_gt.to_csv(OUTPUT_PATH, index=False)
            time.sleep(2)
            continue

    result = call_groq(EDUCATION_PROMPT, wiki_text, max_retries=3)

    if isinstance(result, dict):
        raw = result.get("raw", result.get("error", ""))
        df_gt.at[idx, "education_text"] = raw.strip() if raw else "error"
    else:
        df_gt.at[idx, "education_text"] = str(result).strip()

    df_gt.to_csv(OUTPUT_PATH, index=False)
    time.sleep(2)

print("\n✓ Education extraction complete — please manually verify education_text column.")
print(f"✓ Saved to: {OUTPUT_PATH}")

Senators needing education extraction: 1


Extracting education:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'create_wikipedia_url' is not defined

In [33]:
# ── Evaluation metrics (Liu et al. Section 6.1.4) ───────────────────────────
# Updated: date normalization, gender exact-match, two-digit year correction,
#          partial birthdate credit, NaN-aware scoring throughout.

import ast
import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
from rouge_score import rouge_scorer
import bert_score

GT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")


# ── Name normalization ────────────────────────────────────────────────────────

def normalize_name(name):
    """Lowercase, strip accents, expand common nickname abbreviations."""
    if not name or (isinstance(name, float) and pd.isna(name)):
        return ""
    name = str(name).lower().strip()
    name = "".join(
        c for c in unicodedata.normalize("NFD", name)
        if unicodedata.category(c) != "Mn"
    )
    expansions = {
        r"\bdan\b": "daniel",   r"\btom\b": "thomas",
        r"\bjon\b": "jonathan", r"\bjim\b": "james",
        r"\bbob\b": "robert",   r"\bwill\b": "william",
        r"\bliz\b": "elizabeth",r"\bpat\b": "patrick",
        r"\bbert\b": "albert",  r"\bted\b": "edward",
        r"\bkatie\b": "katherine", r"\bcat\b": "catherine",
        r"\btimothy\b": "tim",  r"\bchristopher\b": "chris",
        r"\banthony\b": "tony",
    }
    for pattern, replacement in expansions.items():
        name = re.sub(pattern, replacement, name)
    return name


def name_match_score(gt_name, pred_name):
    """1.0 if names are identical or >90% similar after normalization, else 0.0."""
    gt_norm   = normalize_name(gt_name)
    pred_norm = normalize_name(pred_name)
    if not gt_norm or not pred_norm:
        return 1.0 if gt_norm == pred_norm else 0.0
    if gt_norm == pred_norm:
        return 1.0
    return 1.0 if SequenceMatcher(None, gt_norm, pred_norm).ratio() > 0.90 else 0.0


def create_normalized_senator_id(name, state):
    normalized = normalize_name(name)
    name_slug  = "_".join(w.capitalize() for w in normalized.split() if w)
    return f"{name_slug}_{state.upper()}"


# ── Fuzzy senator merge ───────────────────────────────────────────────────────

def match_by_fuzzy_name(df_gt, df_pred):
    """Match GT↔predictions by fuzzy name when senator_id direct merge fails."""
    from fuzzywuzzy import fuzz as fuzzy_fuzz, process as fuzzy_process

    results = []
    for _, gt_row in df_gt.iterrows():
        gt_id   = gt_row.get("senator_id", "")
        gt_name = gt_row.get("full_name") or gt_row.get("name", "")

        exact = df_pred[df_pred["senator_id"] == gt_id]
        if not exact.empty:
            for _, pred_row in exact.iterrows():
                results.append({**gt_row, **pred_row})
            continue

        gt_norm    = normalize_name(gt_name)
        pred_names = df_pred["full_name"].fillna("").astype(str).unique()
        matches    = fuzzy_process.extract(
            gt_norm, pred_names, scorer=fuzzy_fuzz.token_sort_ratio, limit=1
        )
        if matches and matches[0][1] > 85:
            matched_name  = matches[0][0]
            pred_matches  = df_pred[df_pred["full_name"] == matched_name]
            for _, pred_row in pred_matches.iterrows():
                results.append({**gt_row, **pred_row})

    return pd.DataFrame(results) if results else pd.DataFrame()


# ── Date normalization ────────────────────────────────────────────────────────

_CURRENT_YEAR = 2025  # used for two-digit year correction

def _two_digit_year_fix(ts, gt_year=None):
    """
    pandas infers two-digit years with a fixed 50-year pivot (≥50 → 1900s).
    That misinterprets e.g. '82' as 1982 (correct) but '54' as 1954 (wrong
    if the senator was born in 1954 — actually correct — but '72' as 1972).
    Strategy: if gt_year is provided, trust the GT year; otherwise accept
    pandas inference only if the year is plausible for a living senator
    (born 1920–2000).
    """
    if pd.isna(ts):
        return ts
    year = ts.year
    # If pandas mapped a two-digit year to future (e.g. 2082), correct it
    if year > _CURRENT_YEAR:
        year -= 100
        ts = ts.replace(year=year)
    return ts


def parse_date(val, gt_year=None):
    """
    Parse a date string in any common format; return pd.Timestamp or NaT.
    Applies two-digit year correction relative to gt_year when available.
    """
    if pd.isna(val) or str(val).strip() in ("", "nan", "None"):
        return pd.NaT
    for fmt in ("%Y-%m-%d", "%m/%d/%y", "%m/%d/%Y", "%B %d, %Y", "%b %d, %Y"):
        try:
            ts = pd.to_datetime(str(val).strip(), format=fmt)
            return _two_digit_year_fix(ts, gt_year)
        except ValueError:
            continue
    ts = pd.to_datetime(str(val), errors="coerce")
    return _two_digit_year_fix(ts, gt_year) if not pd.isna(ts) else ts


def birthdate_scores(gt_val, pred_val):
    """
    Returns a dict with exact, year_match, month_match scores.
    All are NaN when either value is missing; this prevents missing data
    from being silently scored as 0 and deflating accuracy.
    """
    nan_result = {"exact": float("nan"), "year": float("nan"), "month": float("nan")}

    gt_ts   = parse_date(gt_val)
    # Pass GT year as hint so two-digit LLM years are corrected relative to GT
    gt_year = gt_ts.year if not pd.isna(gt_ts) else None
    pred_ts = parse_date(pred_val, gt_year=gt_year)

    if pd.isna(gt_ts) or pd.isna(pred_ts):
        return nan_result

    return {
        "exact": float(gt_ts == pred_ts),
        "year":  float(gt_ts.year  == pred_ts.year),
        "month": float(gt_ts.month == pred_ts.month),
    }


# ── Gender normalization ──────────────────────────────────────────────────────

def gender_match_score(gt_val, pred_val):
    """
    Exact match after lowercasing. Returns NaN when GT is missing — do not
    penalize extraction for attributes we cannot verify.
    """
    if pd.isna(gt_val) or str(gt_val).strip() == "":
        return float("nan")
    if pd.isna(pred_val) or str(pred_val).strip() == "":
        return 0.0
    return float(str(gt_val).strip().lower() == str(pred_val).strip().lower())


def get_religion_category(religion_str):
    """
    Normalize religion string and return its category from the hierarchy.
    Returns None if the string is effectively missing.
    """
    if pd.isna(religion_str) or str(religion_str).strip() == "":
        return None
    
    norm = str(religion_str).strip().lower()
    
    # Direct lookup
    if norm in RELIGION_HIERARCHY:
        return RELIGION_HIERARCHY[norm]
    
    # Substring matching for common phrases
    for key, category in RELIGION_HIERARCHY.items():
        if key in norm or norm in key:
            return category
    
    # Default: treat as its own category
    return norm


def religion_match_score(gt_val, pred_val):
    """
    Hierarchical religion matching:
      - 1.0 for exact match (after lowercasing)
      - 0.7 for parent-child match (e.g., Methodist vs Christian)
      - 0.0 for unrelated religions
      - NaN when either GT or pred is missing (do not score as incorrect)
    
    This allows partial credit when extracting a parent category (Christian)
    when GT is a specific denomination (Methodist), reflecting that the
    extraction is partially correct.
    """
    if pd.isna(gt_val) or str(gt_val).strip() == "":
        return float("nan")
    if pd.isna(pred_val) or str(pred_val).strip() == "":
        return float("nan")  # <-- key difference from gender_match_score
    
    gt_norm = str(gt_val).strip().lower()
    pred_norm = str(pred_val).strip().lower()
    
    # Exact match
    if gt_norm == pred_norm:
        return 1.0
    
    # Hierarchical match
    gt_cat = get_religion_category(gt_norm)
    pred_cat = get_religion_category(pred_norm)
    
    if gt_cat and pred_cat:
        if gt_cat == pred_cat:
            # Same category but different names — partial credit
            return 0.7
    
    # No match
    return 0.0


# ── Parsing helpers for structured fields ─────────────────────────────────────

def parse_committee_roles(s):
    if not s or (isinstance(s, float) and pd.isna(s)):
        return ""
    try:
        s_str = str(s).strip()
        if s_str.startswith("["):
            roles = ast.literal_eval(s_str)
            return "|".join(roles) if isinstance(roles, list) else str(s)
    except Exception:
        pass
    return str(s)


def parse_education(s):
    if not s or (isinstance(s, float) and pd.isna(s)):
        return ""
    try:
        s_str = str(s).strip()
        if s_str.startswith("["):
            edu_list = ast.literal_eval(s_str)
            institutions = [
                str(edu["institution"])
                for edu in edu_list
                if isinstance(edu, dict) and edu.get("institution")
            ]
            return "|".join(institutions)
    except Exception:
        pass
    return str(s)


def parse_education_detailed(s, format_type="auto"):
    """
    Parse education data into structured list of (degree, institution, year) tuples.
    Handles both JSON format (LLM output) and pipe-delimited format (GT).
    Returns list of dicts with keys: degree, institution, year.
    """
    if not s or (isinstance(s, float) and pd.isna(s)):
        return []
    
    s_str = str(s).strip()
    
    # Try JSON format first (LLM output)
    if s_str.startswith("["):
        try:
            edu_list = ast.literal_eval(s_str)
            education_items = []
            for edu in edu_list:
                if isinstance(edu, dict):
                    degree = str(edu.get("degree", "")).strip() if edu.get("degree") else None
                    institution = str(edu.get("institution", "")).strip() if edu.get("institution") else None
                    year = str(edu.get("year", "")).strip() if edu.get("year") else None
                    if institution:  # Only include if we have at least institution
                        education_items.append({
                            "degree": degree,
                            "institution": institution,
                            "year": year
                        })
            return education_items
        except Exception:
            pass
    
    # Try pipe-delimited format (GT: "degree|institution|year|degree2|institution2|year2")
    if "|" in s_str:
        try:
            parts = [p.strip() for p in s_str.split("|")]
            education_items = []
            # Process in groups of 3 (degree, institution, year)
            for i in range(0, len(parts), 3):
                if i + 1 < len(parts):  # At least need degree and institution
                    degree = parts[i] if i < len(parts) and parts[i] else None
                    institution = parts[i + 1] if i + 1 < len(parts) and parts[i + 1] else None
                    year = parts[i + 2] if i + 2 < len(parts) and parts[i + 2] else None
                    
                    # Skip placeholder years like "no year given"
                    if year and "no year" in year.lower():
                        year = None
                    
                    if institution:  # Only include if we have at least institution
                        education_items.append({
                            "degree": degree,
                            "institution": institution,
                            "year": year
                        })
            return education_items
        except Exception:
            pass
    
    return []


def normalize_degree(deg):
    """Normalize degree strings for comparison (B.S. → BS, B.A. → BA, etc.)."""
    if not deg or pd.isna(deg):
        return ""
    deg = str(deg).upper().strip()
    # Remove periods and extra spaces
    deg = deg.replace(".", "").replace(",", "").replace("'", "")
    
    # Define mappings with priority (longer/more specific first)
    # Check exact matches first, then substring matches
    exact_matches = {
        "MBA": "MBA",
        "JD": "JD",
        "BS": "BS",
        "BA": "BA",
        "MA": "MA",
        "MS": "MS",
        "MIA": "MIA",    # Master of International Affairs
        "PHD": "PHD",
    }
    
    # If exact match, return immediately
    if deg in exact_matches:
        return exact_matches[deg]
    
    # Substring/phrase mappings (longer patterns first to avoid partial matches)
    substring_mappings = [
        ("BACHELOR OF SCIENCE", "BS"),
        ("BACHELOR OF ARTS", "BA"),
        ("BACHELORS SCIENCE", "BS"),
        ("BACHELORS ARTS", "BA"),
        ("BACHELOR DEGREE", "BA"),
        ("BACHELORS DEGREE", "BA"),
        ("JURIS DOCTOR", "JD"),
        ("JURIS DOCTORATE", "JD"),
        ("MASTER OF BUSINESS", "MBA"),  # MBA must come before MA
        ("MASTER OF SCIENCE", "MS"),
        ("MASTER OF ARTS", "MA"),
        ("DOCTOR OF PHILOSOPHY", "PHD"),
        ("DOCTORATE", "PHD"),
    ]
    
    for pattern, result in substring_mappings:
        if pattern in deg:
            return result
    
    return deg


def normalize_school(school):
    """Normalize school names for fuzzy matching."""
    if not school or pd.isna(school):
        return ""
    school = str(school).lower().strip()
    # Remove common suffixes
    for suffix in ["university", "college", "school", "institute", "of", "the"]:
        school = school.replace(suffix, "").strip()
    return school


def compare_education_components(gt_items, pred_items):
    """
    Compare education detail by detail: degree, institution, year.
    Returns dict with per-component match counts and a combined score.
    """
    if not gt_items or not pred_items:
        return {
            "degree_exact": float("nan"), "institution_fuzzy": float("nan"), 
            "year_exact": float("nan"), "combined_score": float("nan")
        }
    
    # Simple matching: compare first degree from each (or allow multiple comparisons)
    degree_matches = 0
    school_matches = 0
    year_matches = 0
    
    # Compare degrees
    if gt_items[0].get("degree") and pred_items[0].get("degree"):
        gt_deg = normalize_degree(gt_items[0]["degree"])
        pred_deg = normalize_degree(pred_items[0]["degree"])
        degree_matches = float(gt_deg == pred_deg)
    
    # Compare institutions (fuzzy)
    if gt_items[0].get("institution") and pred_items[0].get("institution"):
        gt_school = normalize_school(gt_items[0]["institution"])
        pred_school = normalize_school(pred_items[0]["institution"])
        # Fuzzy match via substring or SequenceMatcher
        school_matches = 1.0 if (gt_school in pred_school or pred_school in gt_school or 
                                 SequenceMatcher(None, gt_school, pred_school).ratio() > 0.80) else 0.0
    
    # Compare years
    if gt_items[0].get("year") and pred_items[0].get("year"):
        gt_year = str(gt_items[0]["year"]).strip()
        pred_year = str(pred_items[0]["year"]).strip()
        year_matches = float(gt_year == pred_year)
    
    # Combined: average of the three
    components = [degree_matches, school_matches, year_matches]
    combined = sum(components) / len(components) if components else float("nan")
    
    return {
        "degree_exact": degree_matches,
        "institution_fuzzy": school_matches,
        "year_exact": year_matches,
        "combined_score": combined
    }


# ── Main evaluation loop ──────────────────────────────────────────────────────

if not GT_PATH.exists():
    print("ground_truth.csv not found — skipping evaluation.")
else:
    df_gt   = pd.read_csv(GT_PATH)
    df_pred = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
    scorer_r = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)

    for style in ["direct", "pseudocode", "icl"]:
        df_style = df_pred[df_pred["prompt_style"] == style]

        # TRY 1: Direct merge by senator_id
        merged = df_gt.merge(df_style, on="senator_id", how="inner")

        # TRY 2: Fuzzy fallback
        if merged.empty and len(df_style) > 0:
            print(f"⚠️  No direct senator_id matches for {style} — falling back to fuzzy name matching")
            merged = match_by_fuzzy_name(df_gt, df_style)

        if merged.empty:
            print(f"✗ No matches found for {style} — skipping")
            continue

        print(f"\n{'='*60}")
        print(f"  PROMPT STYLE: {style.upper()}  (n={len(merged)})")
        print(f"{'='*60}")

        # ── Full name ─────────────────────────────────────────────
        if "full_name_x" in merged.columns and "full_name_y" in merged.columns:
            name_scores = [
                name_match_score(gt, pred)
                for gt, pred in zip(
                    merged["full_name_x"].fillna(""),
                    merged["full_name_y"].fillna("")
                )
            ]
            print(f"Accuracy   — full_name (fuzzy):  {sum(name_scores)/len(name_scores):.2%}")

        # ── Gender (exact match, NaN-aware) ───────────────────────
        gt_gender_col   = "gender" if "gender" in merged.columns else None
        pred_gender_col = "gender_y" if "gender_y" in merged.columns else (
                          "gender"   if "gender"   in merged.columns else None)

        # Resolve _x/_y suffix collisions from merge
        for candidate in ["GT_gender", "gender_x", "gender"]:
            if candidate in merged.columns:
                gt_gender_col = candidate
                break
        for candidate in ["LLM_gender", "gender_y", "predicted_gender"]:
            if candidate in merged.columns:
                pred_gender_col = candidate
                break

        if gt_gender_col and pred_gender_col:
            g_scores = [
                gender_match_score(gt, pred)
                for gt, pred in zip(merged[gt_gender_col], merged[pred_gender_col])
            ]
            valid = [s for s in g_scores if not pd.isna(s)]
            if valid:
                print(f"Accuracy   — gender (exact):     {sum(valid)/len(valid):.2%}  "
                      f"(n={len(valid)}, skipped {len(g_scores)-len(valid)} missing GT)")

        # ── Birthdate (normalized, partial credit) ────────────────
        gt_bd_col   = next((c for c in ["GT_birthdate", "birthdate_x", "birthdate"] if c in merged.columns), None)
        pred_bd_col = next((c for c in ["LLM_birthdate", "birthdate_y", "predicted_birthdate"] if c in merged.columns), None)

        if gt_bd_col and pred_bd_col:
            bd_results = [
                birthdate_scores(gt, pred)
                for gt, pred in zip(merged[gt_bd_col], merged[pred_bd_col])
            ]
            for metric in ["exact", "year", "month"]:
                valid = [r[metric] for r in bd_results if not pd.isna(r[metric])]
                if valid:
                    print(f"Accuracy   — birthdate_{metric:<6}:  {sum(valid)/len(valid):.2%}  "
                          f"(n={len(valid)}, skipped {len(bd_results)-len(valid)} missing)")

        # ── Religion (hierarchical match) ─────────────────────────
        if "religion" in merged.columns and "religious_affiliation" in merged.columns:
            rel_scores = []
            for gt, pred in zip(merged["religion"], merged["religious_affiliation"]):
                score = religion_match_score(gt, pred)
                rel_scores.append(score)
            valid = [s for s in rel_scores if not pd.isna(s)]
            if valid:
                print(f"Accuracy   — religion:           {sum(valid)/len(valid):.2%}  "
                      f"(n={len(valid)})")

        # ── Rouge-1: text-overlap fields ──────────────────────────
        for field, gt_col, pred_col, parse_fn in [
            ("education",       "education_text",    "education",       parse_education),
            ("committee_roles", "committee_roles_x", "committee_roles_y", parse_committee_roles),
        ]:
            if gt_col not in merged.columns:
                continue
            scores = []
            for _, row in merged.iterrows():
                pred = parse_fn(row.get(pred_col, "") or "")
                gt   = str(row.get(gt_col, "") or "").strip()
                if gt:
                    scores.append(scorer_r.score(gt, pred)["rouge1"].fmeasure)
            if scores:
                print(f"Rouge-1    — {field}: {sum(scores)/len(scores):.3f}")

        # ── BERTScore: semantic similarity ────────────────────────
        for field, gt_col, pred_col, parse_fn in [
            ("education",       "education_text",    "education",       parse_education),
            ("committee_roles", "committee_roles_x", "committee_roles_y", parse_committee_roles),
        ]:
            if gt_col not in merged.columns:
                continue
            preds = [parse_fn(p) for p in merged[pred_col].fillna("").astype(str)]
            gts   = [str(g).strip() for g in merged[gt_col].fillna("").astype(str)]
            if any(gts):
                _, _, F = bert_score.score(preds, gts, lang="en", verbose=False)
                print(f"BERT score — {field}: F1={F.mean().item():.3f}")

        # ── Education component breakdown: degree, institution, year ──────────
        if "education" in merged.columns and "education_text" in merged.columns:
            print("\n── Education Components (Detailed) ──")
            component_scores = {
                "degree_exact": [],
                "institution_fuzzy": [],
                "year_exact": [],
                "combined_score": []
            }
            
            for _, row in merged.iterrows():
                gt_edu = row.get("education_text", "")
                pred_edu = row.get("education", "")
                
                # Parse both into structured format
                try:
                    # Parse BOTH GT and pred using parse_education_detailed to handle
                    # pipe-delimited format (degree|institution|year) correctly
                    gt_items = parse_education_detailed(gt_edu) if gt_edu else []
                    pred_items = parse_education_detailed(pred_edu) if pred_edu else []
                    
                    # Compare components
                    comp_result = compare_education_components(gt_items, pred_items)
                    for key, val in comp_result.items():
                        if not pd.isna(val):
                            component_scores[key].append(val)
                
                except Exception:
                    pass
            
            # Print component-level metrics
            for comp_name in ["degree_exact", "institution_fuzzy", "year_exact"]:
                scores = component_scores[comp_name]
                if scores:
                    avg = sum(scores) / len(scores)
                    print(f"  {comp_name:<20}: {avg:.2%}  (n={len(scores)})")
            
            # Print combined score
            combined = component_scores["combined_score"]
            if combined:
                avg_combined = sum(combined) / len(combined)
                print(f"  {'combined_all_three':<20}: {avg_combined:.2%}  (n={len(combined)})")

        # ── Stratified religion accuracy by signal type ───────────
        if "religion_signal" in merged.columns:
            print("\n── Religion Accuracy by Signal Type ──")
            for signal_val in ["explicit", "not_explicit"]:
                subset = merged[merged["religion_signal"] == signal_val]
                if subset.empty:
                    print(f"  {signal_val}: no data")
                    continue
                if "religion" in subset.columns and "religious_affiliation" in subset.columns:
                    valid = [
                        religion_match_score(gt, pred)
                        for gt, pred in zip(subset["religion"], subset["religious_affiliation"])
                        if not pd.isna(religion_match_score(gt, pred))
                    ]
                    if valid:
                        print(f"  {signal_val:<15}: {sum(valid)/len(valid):.2%}  (n={len(valid)})")



  PROMPT STYLE: DIRECT  (n=99)
Accuracy   — full_name (fuzzy):  81.82%
Accuracy   — gender (exact):     92.71%  (n=96, skipped 3 missing GT)
Accuracy   — birthdate_exact :  70.59%  (n=17, skipped 82 missing)
Accuracy   — birthdate_year  :  88.24%  (n=17, skipped 82 missing)
Accuracy   — birthdate_month :  76.47%  (n=17, skipped 82 missing)
Accuracy   — religion:           64.05%  (n=42)
Rouge-1    — education: 0.385
Rouge-1    — committee_roles: 0.210


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT score — education: F1=0.659


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT score — committee_roles: F1=0.615

── Education Components (Detailed) ──
  degree_exact        : 21.33%  (n=75)
  institution_fuzzy   : 72.00%  (n=75)
  year_exact          : 14.67%  (n=75)
  combined_all_three  : 36.00%  (n=75)

── Religion Accuracy by Signal Type ──
  explicit       : 68.57%  (n=7)
  not_explicit   : 63.14%  (n=35)

  PROMPT STYLE: PSEUDOCODE  (n=99)
Accuracy   — full_name (fuzzy):  88.89%
Accuracy   — gender (exact):     67.71%  (n=96, skipped 3 missing GT)
Accuracy   — birthdate_exact :  80.00%  (n=15, skipped 84 missing)
Accuracy   — birthdate_year  :  100.00%  (n=15, skipped 84 missing)
Accuracy   — birthdate_month :  80.00%  (n=15, skipped 84 missing)
Accuracy   — religion:           68.33%  (n=18)
Rouge-1    — education: 0.378
Rouge-1    — committee_roles: 0.211


...

BERT score — committee_roles: F1=0.560

── Education Components (Detailed) ──
  degree_exact        : 22.22%  (n=72)
  institution_fuzzy   : 77.78%  (n=72)
  year_exact          : 15.28%  (n=72)
  combined_all_three  : 38.43%  (n=72)

── Religion Accuracy by Signal Type ──
  explicit       : 68.57%  (n=7)
  not_explicit   : 68.18%  (n=11)

  PROMPT STYLE: ICL  (n=99)
Accuracy   — full_name (fuzzy):  92.93%
Accuracy   — gender (exact):     92.71%  (n=96, skipped 3 missing GT)
Accuracy   — birthdate_exact :  80.00%  (n=15, skipped 84 missing)
Accuracy   — birthdate_year  :  100.00%  (n=15, skipped 84 missing)
Accuracy   — birthdate_month :  80.00%  (n=15, skipped 84 missing)
Accuracy   — religion:           53.93%  (n=28)
Rouge-1    — education: 0.376
Rouge-1    — committee_roles: 0.184


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT score — education: F1=0.626


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT score — committee_roles: F1=0.605

── Education Components (Detailed) ──
  degree_exact        : 33.80%  (n=71)
  institution_fuzzy   : 71.83%  (n=71)
  year_exact          : 14.08%  (n=71)
  combined_all_three  : 39.91%  (n=71)

── Religion Accuracy by Signal Type ──
  explicit       : 68.57%  (n=7)
  not_explicit   : 49.05%  (n=21)


## 10. Baseline Comparison Table (Liu et al. Tables 4–5)

Aggregates LLM extraction vs regex and spaCy NER across all processed profiles.
Reproduce the paper's key finding: *LLM outperforms traditional methods in almost all scenarios.*

In [37]:
# ── Run baselines across all processed profiles ──────────────────────────
# Reads the HTML files already processed by the main pipeline.
# Uses the same extraction mode (preprocessed vs raw HTML) as the main pipeline
baseline_rows = []
for hf in tqdm(html_files, desc="Baselines"):
    html = hf.read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    regex_r = regex_extract(text)
    spacy_r = spacy_extract(text)
    baseline_rows.append({
        "senator_id":             hf.stem,
        "regex_name":             regex_r["full_name"],
        "regex_email_found":      1 if regex_r["email"] else 0,
        "regex_phone_found":      1 if regex_r["phone"] else 0,
        "spacy_top_person":       spacy_r["persons_detected"][0] if spacy_r["persons_detected"] else None,
        "spacy_orgs_count":       len(spacy_r["orgs_detected"]),
    })

df_bl = pd.DataFrame(baseline_rows)
df_bl.to_csv(OUTPUT_DIR / "baselines.csv", index=False)

# ── Compare coverage: LLM vs baselines ───────────────────────────────────
df_t1  = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
merged_bl = df_t1.merge(df_bl, on="senator_id")

llm_name_cov   = merged_bl["full_name"].notna().mean()
regex_name_cov = merged_bl["regex_name"].notna().mean()
spacy_name_cov = merged_bl["spacy_top_person"].notna().mean()

llm_relig_cov   = merged_bl["religious_affiliation"].notna().mean()

print("=== Name extraction coverage (Liu et al. Table 4–5 analog) ===")
print(f"  LLM:   {llm_name_cov:.1%}")
print(f"  Regex: {regex_name_cov:.1%}")
print(f"  spaCy: {spacy_name_cov:.1%}")
print("\n=== Religious Affiliation extraction coverage ===")
print(f"  LLM:   {llm_relig_cov:.1%}")
print("\nFull baseline results saved to baselines.csv")

Baselines:   0%|          | 0/100 [00:00<?, ?it/s]

=== Name extraction coverage (Liu et al. Table 4–5 analog) ===
  LLM:   100.0%
  Regex: 99.0%
  spaCy: 100.0%

=== Religious Affiliation extraction coverage ===
  LLM:   30.3%

Full baseline results saved to baselines.csv


## 9a. Prompt Ablation Study (Liu et al. Table 13)

A rigorous comparison of all three prompt styles on a **fixed held-out subset of 25 senators**.
Each senator is extracted using direct, pseudocode, and ICL prompts.
This replicates Liu et al.'s prompt-style ablation (Section 4.2, Table 13) to quantify the effect of prompt engineering
on field coverage and accuracy.

**Features:**
- Fixed random seed (42) for reproducibility
- Separate results file (`ablation_results.json`) to keep ablation independent from main pipeline
- Resume-safe — skips already-completed combinations
- 3-second rate limit protection between API calls
- Coverage analysis across all 9 fields per prompt style


In [35]:
# ── Create fixed ablation subset (seed=42 for reproducibility) ────────────
# Use a held-out subset of 25 senators for rigorous prompt comparison
random.seed(42)
ablation_size = 25

# Sample ablation senators from all available files
ablation_senators = random.sample([f.stem for f in html_files], min(ablation_size, len(html_files)))

print(f"🔬 ABLATION SUBSET")
print(f"  Fixed seed:      42")
print(f"  Ablation size:   {len(ablation_senators)}")
print(f"  Senators:\n    {', '.join(ablation_senators[:5])}... (+{len(ablation_senators)-5} more)")
print()


🔬 ABLATION SUBSET
  Fixed seed:      42
  Ablation size:   25
  Senators:
    Roger_Marshall_KS, Catherine_Cortez_Masto_NV, Amy_Klobuchar_MN, Tim_Kaine_VA, James_Lankford_OK... (+20 more)



In [36]:
# ── Define all prompt styles for ablation ───────────────────────────────
ABLATION_STYLES = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# ── Load or initialize ablation results ──────────────────────────────────
ablation_path = OUTPUT_DIR / "ablation_results.json"

if ablation_path.exists():
    with open(ablation_path) as f:
        ablation_results = json.load(f)
    completed = set()
    for sid, styles_dict in ablation_results.items():
        for style in styles_dict.keys():
            completed.add((sid, style))
    print(f"✓ Resuming — {len(completed)} senator-style combinations already processed")
else:
    ablation_results = {}
    completed = set()

# ── Run ablation loop: all 3 prompts on all ablation senators ────────────
ablation_tasks = [(sid, style) for sid in ablation_senators for style in ABLATION_STYLES.keys()]
remaining_tasks = [t for t in ablation_tasks if t not in completed]

print(f"Remaining ablation tasks: {len(remaining_tasks)}/{len(ablation_tasks)}")
print()

for senator_id, style_name in tqdm(remaining_tasks, desc="Ablation loop"):
    # Load HTML and extract text
    html_file = [f for f in html_files if f.stem == senator_id]
    if not html_file:
        continue
    
    html = html_file[0].read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    
    # Extract with current style
    prompt = ABLATION_STYLES[style_name]
    result = call_groq(prompt, text)
    
    # Initialize or update senator's results
    if senator_id not in ablation_results:
        ablation_results[senator_id] = {}
    
    ablation_results[senator_id][style_name] = result
    
    # Save incrementally (resume-safe)
    with open(ablation_path, "w") as f:
        json.dump(ablation_results, f, indent=2)
    
    # Rate limit protection — 3 seconds between calls
    time.sleep(3)

print(f"\n✓ Ablation complete. Results saved to: ablation_results.json")


Remaining ablation tasks: 75/75



Ablation loop:   0%|          | 0/75 [00:00<?, ?it/s]


✓ Ablation complete. Results saved to: ablation_results.json


In [37]:
# ── Flatten ablation results to CSV ──────────────────────────────────────
# Load ablation results
with open(OUTPUT_DIR / "ablation_results.json") as f:
    ablation_results = json.load(f)

# Define all fields to extract
T1_FIELDS_ABLATION = [
    "full_name",
    "birthdate",
    "birth_year_inferred",
    "gender",
    "race_ethnicity",
    "education",
    "committee_roles",
    "religious_affiliation",
    "religious_affiliation_inferred"
]

# Flatten to CSV: one row per (senator, prompt_style) combination
ablation_rows = []
for senator_id, styles_dict in ablation_results.items():
    for style_name, extraction_result in styles_dict.items():
        row = {
            "senator_id": senator_id,
            "prompt_style": style_name
        }
        
        # Check for extraction error
        if "error" in extraction_result:
            row["extraction_error"] = extraction_result.get("error")
            for field in T1_FIELDS_ABLATION:
                row[field] = None
        else:
            for field in T1_FIELDS_ABLATION:
                row[field] = extraction_result.get(field)
        
        ablation_rows.append(row)

df_ablation = pd.DataFrame(ablation_rows)
df_ablation.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)

print(f"✓ Flattened {len(df_ablation)} results to ablation_results.csv")
print(f"  Shape: {df_ablation.shape}")
print(f"\nPrompt style breakdown:")
print(df_ablation["prompt_style"].value_counts().to_string())


✓ Flattened 75 results to ablation_results.csv
  Shape: (75, 11)

Prompt style breakdown:
prompt_style
direct        25
pseudocode    25
icl           25


In [ ]:
# ── Coverage comparison: field coverage per prompt style ─────────────────
print("\n" + "=" * 80)
print("📊 PROMPT ABLATION — FIELD COVERAGE COMPARISON")
print("=" * 80)
print(f"\nAblation subset size: {len(ablation_senators)} senators")
print(f"Ablation styles: {', '.join(ABLATION_STYLES.keys())}")
print()

# Calculate coverage for each field by prompt style
coverage_by_style = {}
for style in ABLATION_STYLES.keys():
    style_data = df_ablation[df_ablation["prompt_style"] == style].copy()

    error_mask = style_data.get("extraction_error", pd.Series(dtype=str)).notna()
    valid_data = style_data[~error_mask]
    n_errors = int(error_mask.sum())
    n_valid = len(valid_data)

    if n_valid == 0:
        print(f"  ⚠ {style}: all {n_errors} results are errors — delete ablation_results.json and rerun")
        continue

    coverage_by_style[style] = {}
    for field in T1_FIELDS_ABLATION:
        filled = int(valid_data[field].notna().sum())
        pct = (filled / n_valid * 100)
        coverage_by_style[style][field] = {"filled": filled, "total": n_valid, "pct": pct}

if not coverage_by_style:
    print("⚠ No valid results to display.")
else:
    # Print coverage table
    print(f"{'Field':<35} | {'DIRECT':^15} | {'PSEUDOCODE':^15} | {'ICL':^15}")
    print("-" * 80)
    for field in T1_FIELDS_ABLATION:
        row_str = f"{field:<35} | "
        for style in ["direct", "pseudocode", "icl"]:
            if style in coverage_by_style:
                c = coverage_by_style[style][field]
                row_str += f"{c['filled']}/{c['total']} ({c['pct']:5.1f}%) | "
            else:
                row_str += f"{'N/A':^15} | "
        print(row_str)
    print("=" * 80)

    # Show average coverage per style
    print("\n📈 OVERALL COVERAGE BY PROMPT STYLE:")
    for style in ["direct", "pseudocode", "icl"]:
        if style in coverage_by_style:
            avg_cov = sum(coverage_by_style[style][f]["pct"] for f in T1_FIELDS_ABLATION) / len(T1_FIELDS_ABLATION)
            print(f"  {style:<15}: {avg_cov:6.1f}% (avg across all fields)")
        else:
            print(f"  {style:<15}: N/A (all errors)")

    print("\n💡 KEY OBSERVATIONS:")
    print("  • Compare direct vs pseudocode: Liu et al. find pseudocode slightly better for structured fields")
    print("  • Compare direct/pseudocode vs ICL: Liu et al. find ICL best for occupation-type fields")
    print("    (committee_roles, education, religious_affiliation)")
    print("=" * 80 + "\n")

In [38]:
import json
with open("../outputs/senate_results/ablation_results.json") as f:
    data = json.load(f)

# Check one senator across all 3 styles
first = list(data.keys())[0]
print(first)
print(json.dumps(data[first], indent=2))

Roger_Marshall_KS
{
  "direct": {
    "full_name": "Roger Marshall",
    "birthdate": null,
    "birth_year_inferred": null,
    "gender": null,
    "race_ethnicity": null,
    "education": [
      {
        "degree": "Bachelor\u2019s",
        "institution": "Kansas State University",
        "year": null
      },
      {
        "degree": "Medical Doctorate",
        "institution": "University of Kansas",
        "year": null
      }
    ],
    "committee_roles": [
      "Committee on Agriculture, Nutrition, and Forestry",
      "Chairman of the Subcommittee on Conservation, Forestry, Natural Resources, and Biotechnology",
      "Subcommittee on Food and Nutrition, Specialty Crops, Organics, and Research",
      "Committee on Finance",
      "Committee on Health, Education, Labor, and Pensions",
      "Chairman of the Subcommittee on Primary Health and Retirement Security",
      "Subcommittee on Employment and Workplace Safety",
      "Committee on the Budget"
    ],
    "religious_

# Religion Hierarchy & Hierarchical Matching

To give partial credit for related religions (e.g., Methodist → Christian),
we use a hierarchical taxonomy. Exact matches score 1.0, parent-child matches
(e.g., Methodist vs Christian) score 0.7, and unrelated religions score 0.0.

This better reflects partial correctness: extracting "Christian Methodist" when
GT is "Methodist" is partially correct even if not exact.

## Helper Functions for Evaluation

```python
# Religion hierarchy: maps denominations to their parent category
RELIGION_HIERARCHY = {
    # Catholic tradition
    "catholic": "catholic",
    "roman catholic": "catholic",
    
    # Protestant denominations
    "methodist": "protestant",
    "united methodist": "protestant",
    "methodist church": "protestant",
    "baptist": "protestant",
    "southern baptist": "protestant",
    "presbyterian": "protestant",
    "episcopal": "protestant",
    "episcopalian": "protestant",
    "lutheran": "protestant",
    "evangelical": "protestant",
    "pentecostal": "protestant",
    "assemblies of god": "protestant",
    
    # Broadly Christian (covers generic "Christian" answers)
    "christian": "christian",
    "christian (unspecified)": "christian",
    "protestant": "protestant",
    
    # Other major religions
    "jewish": "jewish",
    "judaism": "jewish",
    "muslim": "muslim",
    "islam": "muslim",
    "jewish orthodox": "jewish",
    "orthodox": "orthodox",
    "mormon": "mormon",
    "church of jesus christ": "mormon",
    "lds": "mormon",
    "unitarian": "unitarian",
    "unitarian universalist": "unitarian",
    
    # Secular / None
    "atheist": "none",
    "agnostic": "none",
    "none": "none",
    "no religion": "none",
    "unaffiliated": "none",
}

def get_religion_category(religion_str):
    """
    Normalize religion string and return its category from the hierarchy.
    """
    if pd.isna(religion_str) or str(religion_str).strip() == "":
        return None
    
    norm = str(religion_str).strip().lower()
    
    # Direct lookup
    if norm in RELIGION_HIERARCHY:
        return RELIGION_HIERARCHY[norm]
    
    # Substring matching for common phrases
    for key, category in RELIGION_HIERARCHY.items():
        if key in norm or norm in key:
            return category
    
    # Default: treat as its own category
    return norm

def religion_match_score(gt_val, pred_val):
    """
    Hierarchical religion matching:
      - 1.0 for exact match (after lowercasing)
      - 0.7 for parent-child match (e.g., Methodist vs Christian)
      - 0.0 for unrelated religions
      - NaN when either GT or pred is missing (do not score as incorrect)
    
    This allows the model to get partial credit when extracting a parent
    category (Christian) when GT is a specific denomination (Methodist),
    reflecting the fact that it's partially correct.
    """
    if pd.isna(gt_val) or str(gt_val).strip() == "":
        return float("nan")
    if pd.isna(pred_val) or str(pred_val).strip() == "":
        return float("nan")
    
    gt_norm = str(gt_val).strip().lower()
    pred_norm = str(pred_val).strip().lower()
    
    # Exact match
    if gt_norm == pred_norm:
        return 1.0
    
    # Hierarchical match
    gt_cat = get_religion_category(gt_norm)
    pred_cat = get_religion_category(pred_norm)
    
    if gt_cat and pred_cat:
        if gt_cat == pred_cat:
            # Same category but different names —  partial credit
            return 0.7
    
    # No match
    return 0.0
```

**Scoring Details:**
- **1.0** — Exact match (Methodist = Methodist)
- **0.7** — Same religious category but different names (Methodist vs Christian, Baptist vs Protestant)
- **0.0** — Different religions (Methodist vs Catholic, Christian vs Muslim)
- **NaN** — Either value missing (not scored)

This approach rewards the model for extracting the correct parent category
when it can't pin down the exact denomination, which is semantically more
reasonable than treating all non-exact matches as completely wrong.